# Le T5 initialise aleatoirement, forme par forme

Ce notebook fait traverser un vrai batch au modele `random_t5_*` du projet, et affiche la forme des
tenseurs a chaque etape. Il suit la section 3.2 de [GUIDE.md](../GUIDE.md), dans le meme ordre.

**C'est exactement l'architecture des deux branches `t5-small`.** `T5ForConditionalGeneration`,
construit depuis la configuration de la revision epinglee par
`configs/experiments/random_t5_100.yaml`. La branche pre-entrainee charge les poids de cette
revision, celle-ci ne les charge pas. Rien d'autre ne les separe.

La troisieme famille du projet, `scratch_*`, est un Transformer ecrit bloc par bloc sous
`src/models/scratch/`. Il porte le meme budget de parametres a 0,11 % pres, mais pas cette
architecture ; la section 2 de [RAPPORT.md](../RAPPORT.md) dit ce que l'ecart recouvre.

**Les poids sont aleatoires.** Le modele est construit ici, il n'est pas entraine : la derniere
cellule ne produit rien de lisible. Pour un modele qui a appris, voir `02_training.ipynb`.

Rien n'est ecrit sur le disque. Le notebook tourne sur CPU en une minute environ.

In [1]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.config import TokenizerConfig, load_pipeline_config
from src.data.dataset import MANIFEST_NAME, load_split
from src.data.example import Example
from src.data.tokenize import IGNORE_INDEX, build_tokenizer, encode_batch
from src.experiments.config import load_experiment_config
from src.models.generation import GenerationConfig
from src.models.pretrained.base import BaselineConfig
from src.models.pretrained.t5 import T5Summarizer
from src.utils.seed import set_seed


def shape(label: str, tensor: torch.Tensor) -> None:
    """Print one tensor shape, aligned."""
    print(f"{label:<34} {tuple(tensor.shape)}")


set_seed(42)
print(f"racine du depot   {REPO_ROOT}")
print(f"torch             {torch.__version__}")

C:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


racine du depot   C:\Users\brill\Projets\Scolaire\Train-vs-Pre-train
torch             2.13.0+cu130


## 1. Deux exemples

On prend deux documents du split de test si le corpus est construit, deux documents ecrits en dur
sinon. Le reste du notebook ne fait pas la difference.

In [2]:
FALLBACK = [
    Example(
        example_id="fallback-0",
        source=(
            "Three men have been arrested on suspicion of murder after a man was found "
            "badly injured at a meat market in the Old Swan area of Liverpool. The victim, "
            "aged 44, was shot in the head and chest and died later in hospital. The three "
            "men remain in police custody for questioning."
        ),
        target=(
            "Three men have been arrested on suspicion of murder over a shooting at a "
            "meat market."
        ),
    ),
    Example(
        example_id="fallback-1",
        source=(
            "The council has approved plans for a new footbridge over the river, replacing "
            "a crossing closed for safety reasons in 2019. Work is expected to start in the "
            "spring and to take eighteen months. Local traders welcomed the decision."
        ),
        target="Plans for a new footbridge over the river have been approved by the council.",
    ),
]

pipeline = load_pipeline_config(REPO_ROOT / "configs" / "data" / "cnn_dailymail.yaml")
processed = Path(pipeline.paths.processed)
if not processed.is_absolute():
    processed = REPO_ROOT / processed

if (processed / MANIFEST_NAME).is_file():
    examples = load_split(processed, "test")[:2]
    print(f"corpus reel, {processed}")
else:
    examples = FALLBACK
    print("corpus absent, exemples de secours (make data pour le construire)")

for example in examples:
    print()
    print(f"  document : {example.source[:110]}...")
    print(f"  reference: {example.target}")

corpus reel, C:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\data\processed\cnn_dailymail

  document : (CNN)It's only June, but 2015 may be remembered as the year the term "transgender" fully entered mainstream co...
  reference: Olympic hero Bruce Jenner appears on Vanity Fair cover as "Caitlyn" Transgender people in the United States are riding an unprecedented wave of visibility . Shows such as "Transparent," "Orange is the New Black" have raised awareness .

  document : Stock market returns have averaged 16 per cent a year under David Cameron's Conservatives party, compared to j...
  reference: Shares have performed nearly twice as well under Tories, new figures show . Stock market returns averaged 16 per cent per year under Conservatives . Returns hovered around nine per cent under Labour and current Coalition .


## 2. Tokenisation

Le tokenizer T5 transforme le texte en entiers. Les trois familles du projet le partagent, donc
l'entree est strictement la meme partout.

Quatre tenseurs sortent de `encode_batch`. `target_ids` porte la cible telle quelle, `labels` sert
a calculer la loss : la difference est le `-100` pose sur le padding, valeur que l'entropie croisee
ignore.

In [3]:
tokenizer_config = TokenizerConfig(
    hf_id=pipeline.tokenizer.hf_id,
    source_prefix=pipeline.tokenizer.source_prefix,
    max_source_tokens=pipeline.tokenizer.max_source_tokens,
    max_target_tokens=pipeline.tokenizer.max_target_tokens,
)
tokenizer = build_tokenizer(tokenizer_config.hf_id)

batch = encode_batch(examples, tokenizer, tokenizer_config)

print(f"tokenizer         {tokenizer_config.hf_id}")
print(f"prefixe source    {tokenizer_config.source_prefix!r}")
print(f"plafond source    {tokenizer_config.max_source_tokens} tokens")
print(f"plafond cible     {tokenizer_config.max_target_tokens} tokens")
print()
shape("input_ids", batch.input_ids)
shape("attention_mask", batch.attention_mask)
shape("target_ids", batch.target_ids)
shape("labels", batch.labels)

print()
print("Les 12 premiers identifiants du document 0 :")
print(" ", batch.input_ids[0, :12].tolist())
print()
print("target_ids contre labels, fin de la sequence 1 :")
print("  target_ids", batch.target_ids[1, -6:].tolist())
print("  labels    ", batch.labels[1, -6:].tolist(), "  <- -100 la ou il y a du padding")
print()
print(
    f"positions comptees par la loss : {int((batch.labels != IGNORE_INDEX).sum())}"
    f" sur {batch.labels.numel()}"
)

tokenizer         t5-small
prefixe source    'summarize: '
plafond source    512 tokens
plafond cible     128 tokens

input_ids                          (2, 512)
attention_mask                     (2, 512)
target_ids                         (2, 59)
labels                             (2, 59)

Les 12 premiers identifiants du document 0 :
  [21603, 10, 41, 254, 17235, 61, 196, 17, 31, 7, 163, 1515]

target_ids contre labels, fin de la sequence 1 :
  target_ids [0, 0, 0, 0, 0, 0]
  labels     [-100, -100, -100, -100, -100, -100]   <- -100 la ou il y a du padding

positions comptees par la loss : 108 sur 118


## 3. Le modele

`T5Summarizer.from_random_init` lit la configuration d'architecture de la revision epinglee et ne
charge pas ses poids. `from_pretrained`, sur la meme revision, les chargerait. C'est la seule
difference entre les experiences `random_t5_*` et `pretrained_ft_*`, et elle se lit dans le champ
`initialization` de la description que le run enregistre.

Le compte de parametres se lit : le quart du modele est la seule table d'embedding, partagee par
l'encodeur, le decodeur et la projection de sortie.

In [4]:
experiment = load_experiment_config(REPO_ROOT / "configs" / "experiments" / "random_t5_100.yaml")
baseline = BaselineConfig.from_tokenizer_config(
    tokenizer_config,
    hf_id=experiment.model.hf_id,
    revision=experiment.model.revision,
)
# eval() desactive le dropout : deux executions de ce notebook donnent le meme resultat.
summarizer = T5Summarizer.from_random_init(baseline)
model = summarizer.model.eval()
config = model.config

for field, value in summarizer.describe().items():
    print(f"{field:<18}{value}")

print()
print(f"d_model           {config.d_model}")
print(f"tetes             {config.num_heads} x {config.d_kv} dimensions")
print(f"couches           {config.num_layers} encodeur, {config.num_decoder_layers} decodeur")
print(f"d_ff              {config.d_ff}")
print(f"vocabulaire       {config.vocab_size}")

embedding_params = config.vocab_size * config.d_model
total = summarizer.num_parameters
print()
print(f"parametres        {total:,}".replace(",", " "))
print(
    f"dont embedding    {embedding_params:,}".replace(",", " ")
    + f"  ({embedding_params / total:.0%})"
)

baseline          t5
hf_id             t5-small
revision          df1b051c49625cf57a3d0d8d3863ed4d13564fe4
mode              untrained
initialization    random
parameters        60506624

d_model           512
tetes             8 x 64 dimensions
couches           6 encodeur, 6 decodeur
d_ff              2048
vocabulaire       32128

parametres        60 506 624
dont embedding    16 449 536  (27%)


## 4. L'embedding

L'embedding transforme `(batch, seq)` entiers en `(batch, seq, d_model)` flottants. Deux details
separent T5 du Transformer de l'article d'origine, et ils comptent pour lire la suite :

- **pas de mise a l'echelle par `sqrt(d_model)`** ;
- **pas d'encodage positionnel ajoute ici**. La position n'entre pas dans le vecteur de token :
  elle arrive a la section 6, comme un biais ajoute aux scores d'attention.

Une seule table sert l'encodeur, le decodeur et la projection de sortie. Ce n'est pas une copie :
c'est le meme objet en memoire.

In [5]:
with torch.no_grad():
    embedded = model.shared(batch.input_ids)

shape("input_ids", batch.input_ids)
shape("embedding", embedded)

print()
print(f"amplitude moyenne de l'embedding : {float(embedded.abs().mean()):.4f}")
print()
print("La table est un seul objet, partage :")
print(f"  encodeur    {model.encoder.embed_tokens.weight is model.shared.weight}")
print(f"  decodeur    {model.decoder.embed_tokens.weight is model.shared.weight}")
print(f"  projection  {model.lm_head.weight is model.shared.weight}")

input_ids                          (2, 512)
embedding                          (2, 512, 512)

amplitude moyenne de l'embedding : 0.7977

La table est un seul objet, partage :
  encodeur    True
  decodeur    True
  projection  True


## 5. Q, K, V et le decoupage en tetes

T5 normalise avant le bloc, pas apres : `T5LayerNorm` est une normalisation RMS, sans recentrage et
sans biais. Viennent ensuite trois projections lineaires du meme tenseur, puis un decoupage en 8
tetes de 64 dimensions.

Le decoupage ne perd ni n'ajoute rien : `512 = 8 x 64`. C'est une operation de forme.

In [6]:
attention = model.encoder.block[0].layer[0].SelfAttention
layer_norm = model.encoder.block[0].layer[0].layer_norm


def split_heads(tensor: torch.Tensor) -> torch.Tensor:
    """Split the last dimension into heads, the way T5 does."""
    rows, length, _ = tensor.shape
    heads = tensor.view(rows, length, attention.n_heads, attention.key_value_proj_dim)
    return heads.transpose(1, 2)


with torch.no_grad():
    normed = layer_norm(embedded)
    query, key, value = attention.q(normed), attention.k(normed), attention.v(normed)
    heads_query, heads_key, heads_value = (split_heads(t) for t in (query, key, value))

shape("entree", embedded)
shape("apres T5LayerNorm", normed)
shape("Q = entree @ W_Q", query)
shape("K = entree @ W_K", key)
shape("V = entree @ W_V", value)
print()
shape("Q decoupe en tetes", heads_query)
shape("K decoupe en tetes", heads_key)
shape("V decoupe en tetes", heads_value)
print()
print(f"{config.d_model} = {attention.n_heads} tetes x {attention.key_value_proj_dim} dimensions")

entree                             (2, 512, 512)
apres T5LayerNorm                  (2, 512, 512)
Q = entree @ W_Q                   (2, 512, 512)
K = entree @ W_K                   (2, 512, 512)
V = entree @ W_V                   (2, 512, 512)

Q decoupe en tetes                 (2, 8, 512, 64)
K decoupe en tetes                 (2, 8, 512, 64)
V decoupe en tetes                 (2, 8, 512, 64)

512 = 8 tetes x 64 dimensions


## 6. Les scores, le biais de position relative et le masque

`Q Kt` donne un score par paire (requete, cle). C'est le seul tenseur dont la taille croit comme le
**carre** de la longueur : c'est lui qui rend la troncature a 512 tokens couteuse.

Deux choses s'y ajoutent avant le softmax.

**Le biais de position relative.** T5 n'a pas d'encodage positionnel absolu. Il apprend un scalaire
par tete et par seau de distance relative, et l'ajoute aux scores. Un seul bloc de l'encodeur le
porte ; les autres reutilisent le sien, donc la notion de position est apprise une seule fois pour
toute la pile.

**Le masque de padding**, qui interdit a toute requete de lire le remplissage. Apres le softmax, le
poids d'une position interdite est nul.

T5 ne divise pas les scores par `sqrt(d_k)` : ce facteur est absorbe par l'initialisation de ses
projections. La cellule refait le calcul a la main et le compare a celui du modele, ce qui verifie
que rien n'a ete oublie en route.

In [7]:
length = batch.input_ids.size(1)
neutral = torch.finfo(torch.float32).min

with torch.no_grad():
    scores = torch.matmul(heads_query, heads_key.transpose(-2, -1))
    position_bias = attention.compute_bias(length, length)
    padding = (1.0 - batch.attention_mask[:, None, None, :].float()) * neutral
    weights = torch.softmax(scores + position_bias + padding, dim=-1)
    context = torch.matmul(weights, heads_value)

shape("scores Q Kt", scores)
shape("biais de position relative", position_bias)
shape("masque de padding additif", padding)
shape("poids apres softmax", weights)
shape("contexte", context)

print()
print(f"somme des poids d'une requete : {float(weights[0, 0, 0].sum()):.4f}  (le softmax normalise)")
print(f"seaux de distance relative    : {config.relative_attention_num_buckets}")
print(f"biais porte par le bloc 0     : {attention.has_relative_attention_bias}")
print(
    "biais porte par le bloc 1     : "
    f"{model.encoder.block[1].layer[0].SelfAttention.has_relative_attention_bias}"
)

with torch.no_grad():
    reference = model.encoder(
        input_ids=batch.input_ids,
        attention_mask=batch.attention_mask,
        output_attentions=True,
    ).attentions[0]

print()
print(f"reconstruction exacte des poids du bloc 0 : {torch.allclose(weights, reference, atol=1e-5)}")

# Les articles de CNN/DailyMail atteignent presque toujours le plafond de 512, auquel cas le batch
# ne porte aucun padding. Deux phrases de longueurs franchement differentes le rendent visible a
# coup sur.
pair = [
    Example(example_id="court", source="A short sentence.", target="Short."),
    Example(
        example_id="long",
        source="A considerably longer sentence, written so that the batch has to be padded.",
        target="Longer.",
    ),
]
small = encode_batch(pair, tokenizer, tokenizer_config)

with torch.no_grad():
    small_weights = model.encoder(
        input_ids=small.input_ids,
        attention_mask=small.attention_mask,
        output_attentions=True,
    ).attentions[0]

real = int(small.attention_mask[0].sum())
width = small.input_ids.size(1)
padded = float(small_weights[0, 0, 0, real:].max())
kept = float(small_weights[0, 0, 0, :real].max())
print()
print(f"batch de demonstration : sequence 0 a {real} vrais tokens sur {width} positions")
print(f"poids max donne au padding par la requete 0 : {padded:.2e}")
print(f"poids max donne aux vrais tokens            : {kept:.2e}")

scores Q Kt                        (2, 8, 512, 512)
biais de position relative         (1, 8, 512, 512)
masque de padding additif          (2, 1, 1, 512)
poids apres softmax                (2, 8, 512, 512)
contexte                           (2, 8, 512, 64)

somme des poids d'une requete : 1.0000  (le softmax normalise)
seaux de distance relative    : 32
biais porte par le bloc 0     : True
biais porte par le bloc 1     : False


reconstruction exacte des poids du bloc 0 : True

batch de demonstration : sequence 0 a 7 vrais tokens sur 19 positions
poids max donne au padding par la requete 0 : 0.00e+00
poids max donne aux vrais tokens            : 2.75e-01


## 7. La sortie multi-tetes

Les tetes sont recomposees, puis `W_O` melange ce qu'elles ont recolte separement. On revient a la
largeur d'entree : un bloc d'attention preserve la forme, ce qui permet de l'empiler.

In [8]:
with torch.no_grad():
    merged = context.transpose(1, 2).contiguous().view(context.size(0), -1, attention.inner_dim)
    attended = attention.o(merged)

shape("contexte par tete", context)
shape("tetes recomposees", merged)
shape("apres W_O", attended)
print()
print(f"forme d'entree preservee : {tuple(embedded.shape) == tuple(attended.shape)}")

contexte par tete                  (2, 8, 512, 64)
tetes recomposees                  (2, 512, 512)
apres W_O                          (2, 512, 512)

forme d'entree preservee : True


## 8. L'encodeur : la memoire

Six blocs identiques, chacun une self-attention puis un feed-forward, tous deux residuels et
pre-normalises. La sortie est la **memoire** : le document lu une fois, que le decodeur consultera
a chacune de ses positions.

In [9]:
with torch.no_grad():
    memory = model.encoder(
        input_ids=batch.input_ids,
        attention_mask=batch.attention_mask,
    ).last_hidden_state

shape("input_ids", batch.input_ids)
shape("memoire de l'encodeur", memory)
shape("masque de la memoire", batch.attention_mask)
print()
print(f"blocs d'encodeur       : {len(model.encoder.block)}")
print(f"sous-couches d'un bloc : {[type(layer).__name__ for layer in model.encoder.block[0].layer]}")

input_ids                          (2, 512)


memoire de l'encodeur              (2, 512, 512)
masque de la memoire               (2, 512)

blocs d'encodeur       : 6
sous-couches d'un bloc : ['T5LayerSelfAttention', 'T5LayerFF']


## 9. Le decalage a droite

Le decodeur ne doit jamais recevoir le token qu'on lui demande de predire. Le modele decale donc la
cible d'une position et prefixe le token de depart.

La cellule affiche les deux sequences l'une sous l'autre : a chaque colonne, le decodeur voit le
token du bas et doit predire celui du haut. Les `-100` des labels redeviennent du padding en
passant dans l'entree du decodeur, qui n'a pas de valeur ignoree.

In [10]:
decoder_input = model.prepare_decoder_input_ids_from_labels(labels=batch.labels)

shape("labels (la cible)", batch.labels)
shape("entree du decodeur", decoder_input)


def render(ids: torch.Tensor, count: int = 7) -> str:
    """Render the first tokens of a sequence as an aligned row."""
    cells = [tokenizer.decode([max(int(value), 0)]).strip() or "_" for value in ids[:count]]
    return " | ".join(f"{cell:<10}" for cell in cells)


ignored = int((batch.labels == IGNORE_INDEX).sum())
print()
print(f"{'labels':<16}", render(batch.labels[0]))
print(f"{'entree decodeur':<16}", render(decoder_input[0]))
print()
print("Le decodeur predit la ligne du haut a partir de la ligne du bas.")
print(f"token de depart : {config.decoder_start_token_id} (le padding : le tokenizer T5 n'a pas")
print("                  de token de debut de sequence)")
print(f"labels ignores  : {ignored} positions a {IGNORE_INDEX}, dont il reste")
print(f"                  {int((decoder_input == IGNORE_INDEX).sum())} dans l'entree du decodeur")

labels (la cible)                  (2, 59)


entree du decodeur                 (2, 59)

labels           Olympic    | her        | o          | Bruce      | Jen        | ner        | appears   
entree decodeur  <pad>      | Olympic    | her        | o          | Bruce      | Jen        | ner       

Le decodeur predit la ligne du haut a partir de la ligne du bas.
token de depart : 0 (le padding : le tokenizer T5 n'a pas
                  de token de debut de sequence)
labels ignores  : 10 positions a -100, dont il reste
                  0 dans l'entree du decodeur


## 10. Le decodeur et les logits

Chaque bloc du decodeur enchaine trois sous-couches : self-attention causale, cross-attention sur
la memoire, feed-forward. La self-attention y construit son biais de position en mode causal :
aucune distance positive, donc rien qui parle de l'avenir.

La projection finale envoie chaque position sur une entree du vocabulaire par score. Ce sont des
**logits** : pas encore des probabilites. Comme la table est partagee avec l'embedding, `t5-small`
remet l'echelle en multipliant par `d_model ** -0.5` avant la projection.

In [11]:
with torch.no_grad():
    hidden = model.decoder(
        input_ids=decoder_input,
        encoder_hidden_states=memory,
        encoder_attention_mask=batch.attention_mask,
    ).last_hidden_state
    logits = model.lm_head(hidden * (config.d_model**-0.5))
    probabilities = logits.softmax(dim=-1)

shape("etats du decodeur", hidden)
shape("logits", logits)
shape("probabilites", probabilities)

print()
print(f"sous-couches d'un bloc : {[type(layer).__name__ for layer in model.decoder.block[0].layer]}")

best = int(logits[0, 0].argmax())
print()
print(f"token le plus probable en position 0 : {best} = {tokenizer.decode([best])!r}")
print(f"sa probabilite       : {float(probabilities[0, 0, best]):.5f}")
print(f"probabilite uniforme : {1 / config.vocab_size:.5f}")
print("Le modele a bien une preference, mais elle vient de son initialisation :")
print("il n'a rien lu du document, et la section 11 le mesure.")

etats du decodeur                  (2, 59, 512)
logits                             (2, 59, 32128)
probabilites                       (2, 59, 32128)

sous-couches d'un bloc : ['T5LayerSelfAttention', 'T5LayerCrossAttention', 'T5LayerFF']

token le plus probable en position 0 : 0 = '<pad>'
sa probabilite       : 0.02404
probabilite uniforme : 0.00003
Le modele a bien une preference, mais elle vient de son initialisation :
il n'a rien lu du document, et la section 11 le mesure.


## 11. La loss

Entropie croisee entre les logits et les labels, les deux aplatis en une prediction par ligne. Les
positions de padding portent `-100` et sont retirees de la moyenne.

Donner `labels` suffit : le modele fabrique lui-meme l'entree du decodeur, exactement comme la
section 9 l'a fait a la main. La cellule le verifie en comparant les logits.

Un modele non entraine ne sait rien du token suivant, donc sa loss se tient autour de
`ln(vocabulaire)`. Franchement en dessous signalerait une fuite : le decodeur verrait le token
qu'on lui demande. Franchement au-dessus signalerait un masque casse. C'est le controle de sante le
plus simple d'un modele sequence a sequence.

In [12]:
with torch.no_grad():
    output = model(
        input_ids=batch.input_ids,
        attention_mask=batch.attention_mask,
        labels=batch.labels,
    )

print(f"loss mesuree      {float(output.loss):.4f}")
print(f"ln(vocabulaire)   {math.log(config.vocab_size):.4f}")
print()
print(f"memes logits que la section 10 : {torch.allclose(output.logits, logits, atol=1e-4)}")

loss mesuree      10.8548
ln(vocabulaire)   10.3775

memes logits que la section 10 : True


## 12. La generation

A l'entrainement, toute la cible est donnee d'un coup. A la generation, rien n'est donne : le
modele produit un token, se le redonne, et recommence.

La cellule montre que la boucle autoregressive tourne et rend la bonne forme. Elle passe par
`T5Summarizer.generate_ids`, donc par le chemin que l'evaluation emprunte.

In [13]:
greedy = summarizer.generate_ids(
    batch.input_ids, batch.attention_mask, GenerationConfig(max_new_tokens=12, num_beams=1)
)
beams = summarizer.generate_ids(
    batch.input_ids, batch.attention_mask, GenerationConfig(max_new_tokens=12, num_beams=4)
)

shape("greedy (num_beams=1)", greedy)
shape("beam search (num_beams=4)", beams)

print()
for index, sequence in enumerate(greedy):
    print(f"  greedy {index} : identifiants {sequence.tolist()}")
    print(f"             texte        {tokenizer.decode(sequence, skip_special_tokens=True)!r}")

print()
print("Rien de lisible : le modele n'a pas ete entraine, il repete le token de padding.")
print("La plomberie, elle, fonctionne.")

greedy (num_beams=1)               (2, 12)
beam search (num_beams=4)          (2, 12)

  greedy 0 : identifiants [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
             texte        ''
  greedy 1 : identifiants [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
             texte        ''

Rien de lisible : le modele n'a pas ete entraine, il repete le token de padding.
La plomberie, elle, fonctionne.


## Recapitulatif

```text
input_ids                (2, S)
  embedding              (2, S, 512)      <- sans sqrt(d_model), sans position
  T5LayerNorm            (2, S, 512)
  Q, K, V                (2, S, 512)
  en tetes               (2, 8, S, 64)
  scores                 (2, 8, S, S)     <- quadratique en S
  + biais relatif        (1, 8, S, S)     <- la position entre ici
  contexte               (2, 8, S, 64)
  recompose              (2, S, 512)
memoire                  (2, S, 512)
decalage a droite        (2, T)
logits                   (2, T, 32128)
loss                     scalaire
```

## La suite

- `02_training.ipynb` entraine reellement une experience et trace ses courbes.
- [GUIDE.md](../GUIDE.md), section 3.2, decrit la meme architecture en prose, et la section 4 ce qui
  arrive au batch apres la loss : retropropagation, optimiseur, scheduler, validation, checkpoints.
- `tests/unit/test_pretrained_t5.py` et `tests/unit/test_pretrained_base.py` verifient les
  proprietes montrees ici, et echoueraient si elles etaient fausses.